In [1]:
%pip install numpy pandas 

   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.3 MB 5.1 MB/s eta 0:00:03
   ---------------- ----------------------- 5.0/12.3 MB 17.8 MB/s eta 0:00:01
   ------------------------------------- -- 11.5/12.3 MB 23.2 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 21.9 MB/s  0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ----------------------------------- ---- 8.7/9.7 MB 42.0 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 37.6 MB/s  0:00:00

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   --

Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

A) 30 ngày

B) 90 ngày

C) 180 ngày

D) 365 ngày

In [6]:
import pandas as pd

DATA = '../dataset/Transaction/orders.csv'
df = pd.read_csv(DATA)

# 1. Lọc chỉ lấy những đơn hàng 'delivered'
df = df[df['order_status'] == 'delivered']

# 2. Chuyển đổi sang datetime
df['order_date'] = pd.to_datetime(df['order_date'])

# 3. Loại bỏ các đơn hàng mua cùng ngày của cùng 1 khách hàng (nếu bạn muốn tính gap giữa các NGÀY mua khác nhau)
# Bỏ dòng này nếu bạn vẫn muốn coi 2 order trên cùng 1 ngày có gap = 0
df = df.drop_duplicates(subset=['customer_id', 'order_date'])

# 4. Sắp xếp theo khách hàng và ngày mua
df = df.sort_values(by=['customer_id', 'order_date'])

# 5. Lọc những khách hàng có từ 2 đơn hàng (delivered) trở lên
counts = df['customer_id'].value_counts()
khach_hang_nhieu_don = counts[counts >= 2].index
df = df[df['customer_id'].isin(khach_hang_nhieu_don)]

# 6. Tính khoảng cách số ngày giữa 2 lần mua liên tiếp cho từng khách hàng
df['gap'] = df.groupby('customer_id')['order_date'].diff().dt.days

# 7. Lấy trung vị
median_gap = df['gap'].median()
mean_gap = df['gap'].mean()

print(f"Trung vị (Median) số ngày giữa 2 lần mua liên tiếp (tính riêng status delivered): {median_gap} ngày")
print(f"Trung bình (Mean) số ngày giữa 2 lần mua liên tiếp (tính riêng status delivered): {mean_gap:.2f} ngày")


Trung vị (Median) số ngày giữa 2 lần mua liên tiếp (tính riêng status delivered): 178.0 ngày
Trung bình (Mean) số ngày giữa 2 lần mua liên tiếp (tính riêng status delivered): 327.58 ngày


Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price − cogs)/price?

A) Premium

B) Performance

C) Activewear

**D) Standard**

In [7]:
import pandas as pd

# Đọc file dataset
df_products = pd.read_csv('../dataset/Master/products.csv')

# Tính tỷ suất lợi nhuận gộp cho từng sản phẩm: (price - cogs) / price
df_products['gross_margin'] = (df_products['price'] - df_products['cogs']) / df_products['price']

# Tính trung bình theo từng phân khúc (segment) và sắp xếp giảm dần
margin_by_segment = df_products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)

print(margin_by_segment)


segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64


Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

A) defective

**B) wrong_size**

C) changed_mind

D) not_as_described

In [9]:
# Đọc dữ liệu
df_returns = pd.read_csv('../dataset/Transaction/returns.csv')
df_products = pd.read_csv('../dataset/Master/products.csv')

# Nối bảng trả hàng (returns) với bảng sản phẩm (products) qua product_id
df_joined = pd.merge(df_returns, df_products, on='product_id')

# Lọc chỉ những sản phẩm thuộc danh mục "Streetwear"
df_streetwear = df_joined[df_joined['category'] == 'Streetwear']

# Đếm số lượng theo nhóm lý do trả hàng
reason_counts = df_streetwear['return_reason'].value_counts()
print(reason_counts)


return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?
A) organic_search

B) paid_search

C) email_campaign

D) social_media